# Session 1 · Part 1 — Foundations of Distributed Computing

**Practical Distributed Quantum Computing** — IEEE Quantum Week (QCE26)
*Programming distributed applications with NetQMPI over the CUNQA emulator*

**13:00 – 13:30**

---

Distributed quantum computing is, first and last, *distributed computing*. The
constraints are harsher and one operation is outright forbidden by physics, but
the vocabulary, the failure modes and the cost model all carry over from HPC.

This first notebook builds that vocabulary on the machine you are actually
sitting in front of. Nothing here is a slide: every claim is something you can
run and check.

| Notebook | Topic | Time |
|---|---|---|
| **01** (this one) | Anatomy of a distributed system, HPC clusters, decomposition | 13:00–13:30 |
| 02 | MPI: SPMD, point-to-point, collectives | 13:30–14:00 |
| 03 | NetQMPI: quantum directives, non-local gates | 14:00–14:30 |

### How to work through these notebooks

Cells marked **Example** are ready to run — read them, run them, look at the
output. Cells marked **Exercise** have `TODO` markers you must fill in before
they will work. Solutions are in `solutions.ipynb`, but resist opening it: the
exercises are small on purpose.

## 0 · One-time setup

The container runs as `root` because `slurmd` requires it, and Open MPI refuses
to launch as root unless you say so explicitly. If your image already sets these
two variables, this cell is a harmless no-op.

In [3]:
import os

os.environ["OMPI_ALLOW_RUN_AS_ROOT"] = "1"
os.environ["OMPI_ALLOW_RUN_AS_ROOT_CONFIRM"] = "1"

# Everything we launch from a notebook cell inherits this environment.
print("OK — Open MPI will now launch as root")

OK — Open MPI will now launch as root


> **Use `python3`, never `python`.** The image installs `python3` but does not
> provide a `/usr/bin/python` alias. `mpirun ... python script.py` fails, and
> the way it fails is quiet enough to waste ten minutes.

---
## 1 · Anatomy of a distributed system

A supercomputer is organised in four nested levels. From the inside out:

| Level | What it is | Communication inside it |
|---|---|---|
| **Core** | The execution unit: registers, L1/L2 cache | — |
| **Socket** | Tens of cores sharing a last-level cache and a memory controller | Shared cache |
| **Node** | One or more sockets, DRAM, accelerators (GPU — and soon QPU) | **Shared memory**, implicit |
| **Cluster** | Nodes joined by a low-latency interconnect | **Message passing**, explicit |

The hinge of this entire tutorial is the last row. **The node boundary is the
address-space boundary.** Inside a node, two threads communicate by writing to a
variable. Across nodes, there is no such thing: you must package a message, send
it, and have someone receive it.

Let's look at the machine we are on.

In [5]:
# EXAMPLE — the hardware this container can see
!echo "Hostname:      $(hostname)"
!echo "Logical CPUs:  $(nproc)"
!lscpu | grep -E "^(Model name|Socket|Core|Thread|NUMA node\(s\))" || true
!echo
!free -h | head -2

Hostname:      e6d6f0516bc7
Logical CPUs:  12
Model name:                              Intel(R) Core(TM) i7-8700K CPU @ 3.70GHz
Thread(s) per core:                      2
Core(s) per socket:                      6
Socket(s):                               1
NUMA node(s):                            1

               total        used        free      shared  buff/cache   available
Mem:            46Gi        19Gi       5.8Gi       758Mi        23Gi        27Gi


Note the number of logical CPUs. That is your entire "cluster" for today: a
single node. Everything we run will use several *processes* on that one node,
which is enough to learn the programming model — the semantics of MPI are
identical whether the ranks sit on one socket or on a thousand nodes.

What differs is the *cost*. Keep that distinction in mind: today you are
learning correctness, not performance.

---
## 2 · Structure of an HPC cluster

You never run a job on a supercomputer by typing a command and hoping. A
**resource manager** stands between you and the hardware: you describe what you
need, it queues your request, allocates nodes, and launches your processes.

This container runs a real **Slurm** installation — a single-node cluster, but
the same daemons, the same commands, the same semantics as a production system.
This matters more than it might seem: CUNQA's virtual QPUs are launched *as
Slurm jobs*, so this is not scaffolding for the classical half. It is the
mechanism the quantum half runs on.

In [7]:
# EXAMPLE — the cluster, as Slurm sees it
!sinfo
print("-" * 60)
!scontrol show node | head -12

PARTITION AVAIL  TIMELIMIT  NODES  STATE NODELIST
debug*       up   infinite      1   idle e6d6f0516bc7
------------------------------------------------------------
NodeName=e6d6f0516bc7 Arch=x86_64 CoresPerSocket=6 
   CPUAlloc=0 CPUEfctv=12 CPUTot=12 CPULoad=1.59
   AvailableFeatures=(null)
   ActiveFeatures=(null)
   Gres=(null)
   NodeAddr=e6d6f0516bc7 NodeHostName=e6d6f0516bc7 Version=24.11.3
   OS=Linux 6.8.0-111-generic #111~22.04.1-Ubuntu SMP PREEMPT_DYNAMIC Tue Apr 14 17:13:45 UTC  
   RealMemory=48116 AllocMem=0 FreeMem=5749 Sockets=1 Boards=1
   State=IDLE ThreadsPerCore=2 TmpDisk=0 Weight=1 Owner=N/A MCS_label=N/A
   Partitions=debug 
   BootTime=2026-07-31T10:34:56 SlurmdStartTime=2026-09-08T12:56:58
   LastBusyTime=2026-09-08T12:56:58 ResumeAfterTime=None


The three concepts to take away:

- **Partition** (`debug` here) — a queue with a policy attached.
- **Node** — the unit of allocation. `State=IDLE` means it is free.
- **Job** — your request, with a node count, a task count and a time limit.

`srun` runs something *now* inside an allocation; `sbatch` submits a script to
the queue and returns immediately.

In [8]:
# EXAMPLE — one job, four tasks. Note the output order.
!srun --ntasks=4 hostname

e6d6f0516bc7
e6d6f0516bc7
e6d6f0516bc7
e6d6f0516bc7


Four tasks, four lines, and Slurm gave them to us as four independent processes.
They are not yet an MPI program — they cannot talk to each other — but the
launch mechanism is exactly the one MPI builds on.

---
## 3 · The cost of communication

One formula to carry into the rest of the tutorial. Sending an *n*-byte message
between two processes costs, to first order:

$$T(n) = \alpha + \beta n$$

- $\alpha$ — **latency**: the fixed cost per message, whatever its size.
- $\beta$ — the inverse of **bandwidth**: the marginal cost per byte.

The consequence is blunt and counterintuitive: **a thousand 1 kB messages are
dramatically worse than one 1 MB message**, even though the volume is identical.
You pay $\alpha$ a thousand times instead of once.

Hold on to this. In the quantum half, $\alpha$ becomes the time to generate and
confirm an entangled pair — and it is *enormous* compared to a local gate. Every
design decision in NetQMPI follows from that one fact.

---
## Exercise 0 — your first parallel launch

Run the cell below. It is the launch pattern you will use for the rest of the
session: `%%writefile` puts a script on disk, and the next cell launches N
copies of it.

In [24]:
%%writefile check_mpi.py
from mpi4py import MPI

comm = MPI.COMM_WORLD
print(f"rank {comm.Get_rank()} of {comm.Get_size()} reporting in", flush=True)

Writing check_mpi.py


In [25]:
!mpirun -n 4 python3 -u check_mpi.py

rank 3 of 4 reporting in
rank 1 of 4 reporting in
rank 0 of 4 reporting in
rank 2 of 4 reporting in


**Two things to notice, both deliberate.**

**The lines come out in a random order.** Run the cell again — the order will
change. This is not a bug. They are four independent OS processes writing to the
same terminal with no synchronisation whatsoever. MPI does not order standard
output, and if you need ordering you must build it yourself. Internalise this
now; it will save you from a confusing debugging session later.

**Ranks are numbered from 0.** Rank 0 is a perfectly ordinary rank. By universal
convention it plays coordinator when a program needs one, but the MPI standard
grants it no privilege at all.

**You are ready for notebook 02.**